# Fichier `src/cesipath/__init__.py`

Ce notebook liste tout ce que le package `cesipath` expose en import direct. Il sert de carte d'entree pour un developpeur qui decouvre le projet.

Un bon `__init__.py` n'est pas seulement une commodite. C'est une promesse d'API et une facade au sens des patrons de conception [1] : les objets exportes ici sont ceux que les notebooks, scripts et futurs utilisateurs sont invites a importer sans connaitre l'organisation interne des fichiers.


## 1. Regle d'architecture

`__init__.py` re-exporte les objets metier **sans** effet de bord. En particulier :

- pas d'I/O ;
- pas de generation d'instance ;
- pas de tirage aleatoire ;
- pas d'ouverture de figure matplotlib ;
- pas d'execution de benchmark.

Cela rend `import cesipath` rapide, previsible et sans surprise, en restant dans le fonctionnement standard du systeme d'import Python [2]. C'est important dans les notebooks : une cellule d'import doit charger les symboles, pas commencer a calculer.

Le fichier joue aussi un role de facade. Un utilisateur peut ecrire `from cesipath import GraphGenerator, grasp` sans savoir que le premier objet vit dans `graph_generator.py` et le second dans `algorithms/grasp.py`.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import cesipath
print("Nombre d'exports :", len(cesipath.__all__))

## 2. Les primitives graphe

Tout ce qui concerne la construction et la validation d'un graphe statique :

- `GraphGenerationConfig` : parametres et seuils derives ;
- `GraphGenerator` : generateur d'instances ;
- `GraphInstance` : instance complete produite ;
- `EdgeAttributes`, `EdgeStatus` : description des aretes ;
- `InstanceValidator` : garde-fou statique.

Ces exports forment le chemin minimal pour creer un probleme : configurer, generer, verifier, inspecter.

La facade evite d'ecrire des imports trop longs dans les notebooks pedagogiques. On garde les exemples centres sur le concept plutot que sur l'arborescence du package.


In [ ]:
from cesipath import (
    GraphGenerationConfig,
    GraphGenerator,
    GraphInstance,
    EdgeAttributes,
    EdgeStatus,
    InstanceValidator,
)

cfg = GraphGenerationConfig(node_count=6, seed=42)
instance = GraphGenerator(cfg).generate()
print("Valide ?", InstanceValidator(cfg).is_valid(instance))
print("Sommets :", instance.node_count)

## 3. Fermeture metrique

Les primitives Dijkstra et la verification metrique :

- `build_cost_matrix` : dictionnaire d'aretes vers matrice ;
- `complete_graph_with_shortest_paths` : Dijkstra depuis chaque source ;
- `dijkstra` : primitive bas niveau ;
- `check_triangle_inequality` : garde-fou.

Ces fonctions sont exportees parce qu'elles sont utiles au-dela du generateur. Un notebook peut construire un petit graphe a la main, appliquer la fermeture metrique et verifier l'inegalite triangulaire sans instancier tout le pipeline CESIPATH.

En revanche, `reconstruct_path` et `build_neighbor_map` ne sont pas re-exportes ici dans la version actuelle. Ils restent accessibles via `cesipath.metric_closure` si on veut debugger l'interieur de l'algorithme.


In [ ]:
from cesipath import (
    build_cost_matrix,
    check_triangle_inequality,
    complete_graph_with_shortest_paths,
    dijkstra,
)

ok, _ = check_triangle_inequality(instance.completed_costs)
print("Matrice metrique ?", ok)

## 4. Simulation dynamique

Toute la couche dynamique :

- `DynamicNetworkSimulator` : fait evoluer le reseau ;
- `DynamicGraphSnapshot` : etat d'un tour ;
- `DynamicStateValidator` : invariants dynamiques ;
- `sample_dynamic_edge_cost`, `initialize_dynamic_edge_costs`, `refresh_dynamic_edge_costs` : calculs de couts ;
- `dynamic_multiplier`, `DEFAULT_DYNAMIC_SIGMA` : lecture et parametrage.

Cette famille d'exports permet deux usages : lancer une simulation complete, ou tester seulement la variation de cout d'une arete. Les deux niveaux sont utiles dans les notebooks, donc les deux sont exposes.

Le point a garder en tete : le simulateur orchestre, les fonctions de cout calculent. Les exports refletent cette separation.


In [ ]:
from cesipath import (
    DEFAULT_DYNAMIC_SIGMA,
    DynamicGraphSnapshot,
    DynamicNetworkSimulator,
    DynamicStateValidator,
)

simulator = DynamicNetworkSimulator(instance, seed=42)
snap = simulator.initialize_snapshot()
print("sigma par defaut :", DEFAULT_DYNAMIC_SIGMA)
print("Aretes actives   :", snap.active_edge_count)

## 5. Contrat d'entree solveur

Le seul contrat consomme par les algorithmes :

- `SolverInput` : dataclass frozen ;
- `build_static_solver_input` : fabrique depuis une `GraphInstance` ;
- `build_dynamic_solver_input` : fabrique depuis une `GraphInstance` et un `DynamicGraphSnapshot`.

Ces exports sont centraux parce qu'ils decouplent les algorithmes du reseau. Un solveur n'a pas besoin d'importer `GraphInstance`, `DynamicGraphSnapshot` ou les validateurs. Il lit une matrice, des demandes, une capacite et un depot.

Dans une architecture plus large, `SolverInput` serait l'objet le plus facile a serialiser, tester ou fournir a une API externe.


In [ ]:
from cesipath import SolverInput, build_static_solver_input

solver_input = build_static_solver_input(instance)
print(type(solver_input).__name__, solver_input.source)

## 6. Algorithmes (metaheuristiques)

Quatre algorithmes partagent la meme signature `algo(solver_input, **kwargs) -> VRPSolution` :

- `grasp` : construction gloutonne randomisee + recherche locale ;
- `simulated_annealing` : acceptation de type Metropolis [3] + refroidissement geometrique inspire du recuit simule [4] ;
- `tabu_search` : recherche locale avec memoire courte ;
- `genetic_algorithm` : population, croisement OX, mutation et Split.

`VRPSolution` est exporte avec eux parce que c'est le type de retour commun. Il permet aux notebooks et benchmarks d'inspecter `routes`, `total_cost`, `route_count`, etc.

Cette uniformite est ce qui rend les benchmarks simples : on peut mettre les fonctions dans un dictionnaire et les appeler de la meme maniere, avec seulement des hyperparametres differents.


In [ ]:
from cesipath import (
    VRPSolution,
    grasp,
    simulated_annealing,
    tabu_search,
    genetic_algorithm,
)

solution = grasp(solver_input, max_iterations=5, rcl_alpha=0.2, seed=42)
print("Type      :", type(solution).__name__)
print("Tournees  :", solution.route_count)
print("Cout total:", round(solution.total_cost, 2))

## 7. Benchmarks

Outillage pour comparer les algorithmes sur un lot d'instances :

- `run_benchmark(sizes, seeds, algos, algo_kwargs)` ;
- `summarize_benchmark(rows)` ;
- `plot_benchmark_quality`, `plot_benchmark_gap`, `plot_benchmark_runtime` ;
- `save_benchmark_figures(rows)`.

Ces exports ne sont pas necessaires pour resoudre une seule instance, mais ils sont utiles pour produire des resultats repetables. Le benchmark fixe des tailles, des seeds et des algorithmes, puis renvoie des lignes exploitables dans un tableau.

Le fait de les exposer au niveau package encourage une utilisation standardisee : les notebooks de comparaison n'ont pas a recreer chacun leur propre boucle experimentale.


In [ ]:
from cesipath import run_benchmark, summarize_benchmark

rows = run_benchmark(
    sizes=[6, 8],
    seeds=[1, 2],
    algos={"grasp": grasp, "tabu": tabu_search},
    algo_kwargs={
        "grasp": {"max_iterations": 3, "rcl_alpha": 0.2},
        "tabu":  {"max_iterations": 30, "tabu_tenure": 5},
    },
    verbose=False,
)
for s in summarize_benchmark(rows):
    print(s)

## 8. Branche dynamique

Execution et benchmark d'algorithmes sur scenarios dynamiques :

- `DynamicExecution`, `execute_dynamic` : rejoue une solution sur un reseau dynamique et mesure l'ecart planifie / realise ;
- `run_dynamic_benchmark`, `summarize_dynamic_benchmark` ;
- `plot_dynamic_cost_comparison`, `plot_dynamic_gain`, `plot_dynamic_planned_vs_realized` ;
- `save_dynamic_benchmark_figures`.

Cette couche sert a evaluer une question differente du benchmark statique : que gagne-t-on a re-optimiser quand le reseau change ?

Les exports dynamiques restent separes des exports de solveurs. Un algorithme produit une solution ; l'execution dynamique decide comment cette solution se comporte quand les couts et disponibilites evoluent.


In [ ]:
from cesipath import DynamicExecution, execute_dynamic

print("DynamicExecution dispo :", DynamicExecution.__name__)
print("execute_dynamic dispo  :", execute_dynamic.__name__)

## 9. Visualisation algorithmique

Re-exportee depuis `cesipath.algorithms.visualization` :

- `plot_solution(instance, solution)` ;
- `save_solution_plot(...)` : sauvegarde avec numero auto-incremente.

La visualisation du graphe sans solution reste dans `cesipath.visualization` et n'est pas re-exportee ici sous forme de `GraphVisualizer` dans la version actuelle. La raison pratique : les fonctions de trace de solution sont legeres a appeler depuis les notebooks d'algorithmes, tandis que la session interactive du graphe depend davantage du backend matplotlib.

On garde ainsi une facade utile pour les workflows courants, sans forcer tous les utilisateurs a manipuler la classe interactive.


In [ ]:
from cesipath import plot_solution
print("plot_solution dispo :", plot_solution.__name__)

## 10. Resume : ce qu'il faut importer selon le besoin

| Je veux... | J'importe... |
|---|---|
| Generer une instance | `GraphGenerationConfig`, `GraphGenerator` |
| Simuler la dynamique | `DynamicNetworkSimulator` |
| Preparer un solveur | `build_static_solver_input` ou `build_dynamic_solver_input` |
| Resoudre une instance | `grasp`, `simulated_annealing`, `tabu_search`, `genetic_algorithm` |
| Comparer plusieurs solveurs | `run_benchmark`, `summarize_benchmark`, fonctions `plot_*` |
| Evaluer le dynamique | `execute_dynamic`, `run_dynamic_benchmark` |
| Afficher une solution | `plot_solution`, `save_solution_plot` |

A retenir : `__init__.py` est la porte d'entree stable du package. Les sous-modules gardent l'organisation interne ; la facade expose le chemin heureux pour utiliser CESIPATH sans surcharge cognitive.

---

## References

[1] **Gamma, E., Helm, R., Johnson, R. & Vlissides, J. (1994).** *Design Patterns: Elements of Reusable Object-Oriented Software.* Addison-Wesley.

[2] **Python Software Foundation.** *The import system.* Python documentation. https://docs.python.org/3/reference/import.html

[3] **Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H. & Teller, E. (1953).** *Equation of state calculations by fast computing machines.* Journal of Chemical Physics, 21(6), 1087-1092. https://doi.org/10.1063/1.1699114

[4] **Kirkpatrick, S., Gelatt, C. D. & Vecchi, M. P. (1983).** *Optimization by simulated annealing.* Science, 220(4598), 671-680. https://doi.org/10.1126/science.220.4598.671
